In [ ]:
# Install required packages
# IMPORTANT: Do not force-restart the Colab runtime here.
!pip install -q --upgrade langchain langchain-core langchain-community langgraph langchain-openai


In [ ]:
# Load the OpenAI API key from Google Colab Secrets.
# In Colab Secrets, create a secret named exactly: openAiKey

import os
from google.colab import userdata

key = userdata.get("openAiKey")

if not key:
    raise RuntimeError(
        "Colab Secret 'openAiKey' was not found or is empty. "
        "Add the new OpenAI API key to Colab Secrets and enable notebook access."
    )

os.environ["OPENAI_API_KEY"] = key

print("✅ OpenAI API key loaded from Colab Secrets.")
print(f"Prefix: {key[:8]}")
print(f"Suffix: {key[-4:]}")
print(f"Length: {len(key)}")


In [ ]:
# Initialize the OpenAI chat model

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("✅ AI Model Initialized Successfully")


In [ ]:
# Optional connectivity/authentication test

response = llm.invoke(
    "Explain what an HR recruitment agent does in two sentences."
)

print(response.content)


## Setting up State


In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    application: str
    experience_level: str
    skill_match: str
    response: str


## Setting up Start and End


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate


## Define LangGraph Nodes


In [ ]:
def categorize_experience(state: State) -> State:
    print("\nCategorizing the experience level of candidate:")

    prompt = ChatPromptTemplate.from_template(
        "Based on the following job application, categorize the candidate as "
        "'Entry-level', 'Mid-level', or 'Senior-level'. "
        "Respond with exactly one of these three values and nothing else.\n\n"
        "Application: {application}"
    )

    chain = prompt | llm
    experience_level = chain.invoke(
        {"application": state["application"]}
    ).content.strip()

    print(f"Experience Level: {experience_level}")
    return {"experience_level": experience_level}


def assess_skillset(state: State) -> State:
    print("\nAssessing the skillset of candidate:")

    prompt = ChatPromptTemplate.from_template(
        "Assess the candidate's skillset against the job requirement. "
        "The role is a Software Developer role requiring relevant software "
        "development skills. "
        "Respond with exactly 'Match' or 'No Match' and nothing else.\n\n"
        "Application: {application}"
    )

    chain = prompt | llm
    skill_match = chain.invoke(
        {"application": state["application"]}
    ).content.strip()

    print(f"Skill Match: {skill_match}")
    return {"skill_match": skill_match}


def schedule_hr_interview(state: State) -> State:
    print("Scheduling the interview:")
    return {
        "response": "Candidate has been shortlisted for an HR interview."
    }


def escalate_to_recruiter(state: State) -> State:
    print("Escalating to recruiter:")
    return {
        "response": "Candidate has senior-level experience but doesn't match job skills."
    }


def reject_application(state: State) -> State:
    print("Sending rejecting email:")
    return {
        "response": "Candidate doesn't meet JD and has been rejected."
    }


## Define Routing Logic


In [ ]:
def route_app(state: State) -> str:
    # The LLM prompts above force exact values, making these comparisons reliable.
    if state["skill_match"] == "Match":
        return "schedule_hr_interview"
    elif state["experience_level"] == "Senior-level":
        return "escalate_to_recruiter"
    else:
        return "reject_application"


## Build and Compile LangGraph Once


In [ ]:
workflow = StateGraph(State)

workflow.add_node("categorize_experience", categorize_experience)
workflow.add_node("assess_skillset", assess_skillset)
workflow.add_node("schedule_hr_interview", schedule_hr_interview)
workflow.add_node("escalate_to_recruiter", escalate_to_recruiter)
workflow.add_node("reject_application", reject_application)

workflow.add_edge(START, "categorize_experience")
workflow.add_edge("categorize_experience", "assess_skillset")

workflow.add_conditional_edges(
    "assess_skillset",
    route_app,
    {
        "schedule_hr_interview": "schedule_hr_interview",
        "escalate_to_recruiter": "escalate_to_recruiter",
        "reject_application": "reject_application",
    },
)

workflow.add_edge("schedule_hr_interview", END)
workflow.add_edge("escalate_to_recruiter", END)
workflow.add_edge("reject_application", END)

app = workflow.compile()

print("✅ Graph configured and compiled successfully!")


In [ ]:
# Optional: display the graph

from IPython.display import Image, display

display(
    Image(
        app.get_graph().draw_mermaid_png()
    )
)


## Candidate Screening Function


In [ ]:
def run_candidate_screening(application: str):
    results = app.invoke({"application": application})

    return {
        "experience_level": results["experience_level"],
        "skill_match": results["skill_match"],
        "response": results["response"],
    }


## Test with Java Candidate


In [ ]:
application_text = (
    "I have 10 years of experience in software engineering "
    "with expertise in Java, Spring Boot and Microservices."
)

results = run_candidate_screening(application_text)

print("\nComputed Results:")
print(f"Application: {application_text}")
print(f"Experience Level: {results['experience_level']}")
print(f"Skill Match: {results['skill_match']}")
print(f"Response: {results['response']}")


## Test with Python Candidate


In [ ]:
application_text = (
    "I have experience in software engineering "
    "with expertise in Python."
)

results = run_candidate_screening(application_text)

print("\nComputed Results:")
print(f"Application: {application_text}")
print(f"Experience Level: {results['experience_level']}")
print(f"Skill Match: {results['skill_match']}")
print(f"Response: {results['response']}")
